# Track f0 with pYIN

`pyin` follows the fundamental of a monophonic recording frame by frame, then
refits each voiced frame on its harmonics. This notebook runs it on the cello
arco D2 sample, draws the contour on the spectrogram, compares the raw pYIN
grid with the refit, and plays the contour. Read `search_f0` first for the
one-frame `harmonic_f0` search used at the end.

In [ ]:
import time
import numpy as np
import audiospylt as asp

## Load and run `pyin`

The cello plays one note for 8.50 s at 44100 Hz (peak 0.494). `fmin` and `fmax`
are required: C2 to C7 here. Frames use the same length and hop as
`spectrogram` (2048 and 512), so `contour.times` lines up with the spectrogram.
The first call in a process also compiles librosa's decoder, which takes a few
seconds on top of the analysis.

In [ ]:
x, sr = asp.load_audio("../samples/cello_arco_d_sul_c_01.wav", mono=True)
print(f"{len(x) / sr:.2f} s, {sr} Hz, peak {np.max(np.abs(x)):.3f}")
t0 = time.perf_counter()
contour = asp.pyin(x, sr, fmin=65.41, fmax=2093.0)
print(f"pyin {time.perf_counter() - t0:.1f} s")
c256 = asp.pyin(x, sr, fmin=65.41, fmax=2093.0, hop=256)
print(f"frame_length {contour.frame_length}; "
      f"hop 512: {len(contour.times)} frames, step {contour.hop / sr * 1000:.1f} ms; "
      f"hop 256: {len(c256.times)} frames, step {c256.hop / sr * 1000:.1f} ms")
print(f"voiced {int(contour.voiced.sum())}, refined {int(contour.refined.sum())}")
spec = asp.spectrogram(x, sr)
print(f"times == spectrogram.times: {np.array_equal(contour.times, spec.times)}")

733 frames at hop 512 (11.6 ms); hop 256 gives 1465 frames at 5.8 ms. 704
voiced, all 704 refined. The rest of the notebook uses the default hop.

## Voicing

`voiced_prob` never exceeds 0.37 (median 0.18 on voiced frames), yet 704 frames
are voiced. librosa takes the voicing decision from its Viterbi path over
voiced and unvoiced states, not from a threshold on `voiced_prob`. The unvoiced
runs fall at 0.00–0.10 s, 0.28–0.33 s, 0.74–0.79 s, and 8.20–8.29 s.

In [ ]:
vp = contour.voiced_prob[contour.voiced]
print(f"voiced_prob max {np.nanmax(contour.voiced_prob):.2f}, median {np.median(vp):.2f}")
edges = np.diff(np.r_[0, (~contour.voiced).astype(int), 0])
for i, j in zip(np.flatnonzero(edges == 1), np.flatnonzero(edges == -1)):
    print(f"unvoiced {contour.times[i]:.2f}–{contour.times[j - 1]:.2f} s ({j - i} frames)")

## Contour on the spectrogram

Over 1–7 s the median f0 is 74.31 Hz. At the attack (0.34–0.73 s) and the
release (8.30–8.50 s) the contour jumps an octave or more: 147.8–148.5 Hz and
222.2–226.1 Hz across 53 frames; elsewhere it stays 72.5–75.4 Hz. The refit
cannot correct an octave error. `f0_to_tracks` splits the voiced frames into 4
tracks at the unvoiced gaps. With `fmax=600` the octave jumps sit on the second
and third harmonics of the note.

In [ ]:
mid = (contour.times >= 1) & (contour.times <= 7) & contour.voiced
print(f"1–7 s median {np.median(contour.f0[mid]):.2f} Hz ({int(mid.sum())} frames)")
hi = contour.voiced & (contour.f0 > 100)
atk = (contour.times >= 0.34) & (contour.times <= 0.73) & contour.voiced
rel = (contour.times >= 8.30) & (contour.times <= 8.50) & contour.voiced
print(f"octave-error frames: {int(hi.sum())}")
print(f"attack {np.nanmin(contour.f0[atk]):.1f}–{np.nanmax(contour.f0[atk]):.1f} Hz; "
      f"release {np.nanmin(contour.f0[rel]):.1f}–{np.nanmax(contour.f0[rel]):.1f} Hz")
ok = contour.voiced & (contour.f0 <= 100)
print(f"elsewhere {np.nanmin(contour.f0[ok]):.1f}–{np.nanmax(contour.f0[ok]):.1f} Hz")
tracks = asp.f0_to_tracks(contour)
print(f"{tracks.track_id.nunique()} tracks, {len(tracks)} rows")
for tid, g in tracks.groupby("track_id"):
    print(f"  {tid}: {g.time.min():.2f}–{g.time.max():.2f} s, {len(g)} rows, "
          f"{g.freq.min():.1f}–{g.freq.max():.1f} Hz")
asp.nb.show(asp.viz.plot_spectrogram(spec, overlay=tracks, fmax=600, audio=x))

Four tracks: 0.12–0.27 s, 0.34–0.73 s (the octave-high attack), 0.80–8.19 s
(637 rows), and 8.30–8.50 s (the release).

## `refine=False` vs the default

pYIN reports f0 on a 0.1-semitone grid. Over 1–7 s (516 frames) that gives 2
distinct values, 10 cents apart; the refit spreads them. The numbers below are
printed rather than overlaid: at this scale the two contours sit on top of each
other. `help(asp.pyin)` lists `frame_length` and `hop`.

In [ ]:
raw = asp.pyin(x, sr, fmin=65.41, fmax=2093.0, refine=False)
m = (raw.times >= 1) & (raw.times <= 7) & raw.voiced
f_py, f_rf = raw.f0[m], contour.f0[m]
vals, counts = np.unique(np.round(f_py, 3), return_counts=True)
print("pYIN values:", dict(zip(vals.tolist(), counts.tolist())))
print(f"refit distinct (0.001 Hz): {len(np.unique(np.round(f_rf, 3)))}")
med = np.median(f_rf)
iqr = np.subtract(*np.percentile(1200 * np.log2(f_rf / med), [75, 25]))
print(f"refit IQR {iqr:.1f} cents")
diff = 1200 * np.log2(f_py / f_rf)
print(f"refine=False − default: median {np.median(diff):.2f} cents, "
      f"p95 |diff| {np.percentile(np.abs(diff), 95):.1f}, max {np.max(np.abs(diff)):.1f}")
both = contour.voiced & raw.voiced
d_all = 1200 * np.log2(raw.f0[both] / contour.f0[both])
print(f"all {int(both.sum())} voiced: max |diff| {np.max(np.abs(d_all)):.1f} cents")

pYIN: 74.273 Hz (342 frames) and 74.704 Hz (174). Refit: 346 distinct values,
IQR 4.5 cents. Difference (raw minus refit): median 1.75 cents, 95th percentile
of |diff| 10.1 cents, largest 24.5 cents over 1–7 s; over all 704 voiced frames
the largest is 39.4 cents.

## Hear the contour

`amp` is the fundamental's own amplitude. This cello's fundamental is weak and
fades: the note decays from −25 dB RMS at 1 s to −59 dB at 7 s (1 s windows),
and 287 of the 704 voiced frames have `amp == 0` (no peak within 50 cents of
f0), most of them after 4 s. A resynthesis of the tracks is therefore about
14 dB below the recording at 1–3 s and close to silent after 5 s. The second
player holds amplitude at 0.1 so the pitch stays audible. Listen for what the
contour does at the start and the end.

In [ ]:
def rms_db(a):
    return 20 * np.log10(np.sqrt(np.mean(a ** 2)))

print(f"amp == 0: {(tracks.amp == 0).sum()} of {len(tracks)}")
print(f"RMS at 1 s: {rms_db(asp.trim(x, sr, 1, 2)):.0f} dB; at 7 s: {rms_db(asp.trim(x, sr, 7, 8)):.0f} dB")
syn = asp.synthesize(asp.tracks_to_events(tracks), sr)
o13, s13 = asp.trim(x, sr, 1, 3), asp.trim(syn, sr, 1, 3)
print(f"1–3 s: original {rms_db(o13):.1f} dB, contour {rms_db(s13):.1f} dB, "
      f"diff {rms_db(o13) - rms_db(s13):.0f} dB")
print(f"after 5 s, contour RMS {rms_db(asp.trim(syn, sr, 5, 8)):.0f} dB")
asp.nb.play(x, sr)
asp.nb.play(asp.synthesize(asp.tracks_to_events(tracks.assign(amp=0.1)), sr), sr)

The attack and the release jump an octave (and more) above the played D2; the
long middle track stays near 74 Hz.

## One frame: `harmonic_f0`

The same call as in `docs/f0.md` and `search_f0`: peaks from 1.5–2.5 s, search
35–150 Hz. Before you run the cell: how far is the pyin median over that
second from the `harmonic_f0` result?

In [ ]:
peaks = asp.find_peaks(asp.spectrum(asp.trim(x, sr, 1.5, 2.5), sr),
                       min_db=None, prominence_db=10, rel_db=-40)
h = asp.harmonic_f0(peaks, fmin=35, fmax=150)
print(f"{len(peaks)} peaks, f0 {h.f0:.2f} Hz, {h.rms_cents:.2f} cents RMS")
w = (contour.times >= 1.5) & (contour.times <= 2.5) & contour.voiced
med = np.median(contour.f0[w])
print(f"pyin median {med:.2f} Hz ({1200 * np.log2(med / h.f0):+.1f} cents)")

108 peaks, 74.40 Hz at 17.79 cents RMS. The pyin median over the same second is
74.36 Hz, 1.0 cent lower.